# Using Hugging Face Models in Business Scenarios
This notebook demonstrates use cases that can be solved by using **Hugging Face models** and can be run in **Google Colab**.

## What this notebook covers
1. **Sentiment analysis** for customer-review triage  
2. **Zero-shot classification** for support-ticket routing  
3. **Summarization** for executive briefings  
4. **Named Entity Recognition (NER)** for contract / procurement extraction  
5. **Question answering** for policy lookup

---

### How this notebook is structured

Each of the five sections follows the same four-part rhythm:

| Part | What you see |
|------|-------------|
| **Header cell** | Business scenario, task type, and chosen model |
| **Load cell** | Read the demo dataset into a DataFrame |
| **Inference cell(s)** | Create the Hugging Face pipeline and run it |
| **Discussion cell** | Prompts for classroom conversation |

---

### 📌 What is Hugging Face?

Hugging Face is an open-source platform that hosts thousands of pre-trained machine learning models.  
The key concept you need is: **you do not need to train a model from scratch to get useful predictions**.  
Someone else has already trained the model on large amounts of text; you simply *load* it and point it at your data.

The main abstraction we use throughout this notebook is the `pipeline()` function.  
It handles tokenisation, model inference, and output formatting in one line — ideal for understanding the *concept* before going into the mechanics.

### 📌 What is a `pipeline()`?

```
pipeline(task, model=model_id, device=device)
```

- `task` — tells Hugging Face which kind of problem you are solving (e.g. `"sentiment-analysis"`, `"ner"`).
- `model` — the model card ID on the Hugging Face Hub; the weights download automatically on first use.
- `device` — `0` = use GPU (fast), `-1` = use CPU (slower but free).

The pipeline returns a callable object. Pass your text to it and it returns structured predictions.


## Runtime notes
- A **GPU runtime** in Colab is recommended for faster execution, but the examples are intentionally chosen to be feasible on free Colab.
- The first model load for each task can take a little time because weights need to download from Hugging Face.
- Some models have training-data or language limitations, so always review model cards before using them in a real system.

> 📌 **GPU in Colab:**  
> In Google Colab, go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.  
> The free tier gives you a T4 with roughly 15 GB VRAM — more than sufficient for all five models here.  
> If the GPU quota is exhausted, all examples still run on CPU; they will just take 2–4× longer.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
# Google Colab ships with many packages pre-installed, but the Hugging Face
# ecosystem needs to be added explicitly.
#
# Package breakdown:
#   transformers   — the core Hugging Face library; provides pipeline(), model
#                    classes, and tokenisers.
#   datasets       — Hugging Face's data loading library (not heavily used here,
#                    but often needed in companion exercises).
#   accelerate     — handles moving models to GPU automatically; required by
#                    some models when device_map is used.
#   sentencepiece  — a tokenisation library used by several models (e.g. BART).
#   pandas         — tabular data manipulation; we use it to load CSVs and
#                    display results as DataFrames.
#
# The -q flag suppresses the verbose pip output so the cell stays readable.
!pip -q install  datasets accelerate sentencepiece pandas

In [2]:
!pip install "transformers==4.51.3"

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Imports and shared configuration
# ─────────────────────────────────────────────────────────────────────────────
# We set up a small number of constants here that every subsequent section will
# reference.  Keeping shared state at the top of a notebook avoids
# having to re-run scattered setup cells after a kernel restart.

import json                        # used to read .jsonl files line by line
import pandas as pd                # DataFrames for loading and displaying data
from pathlib import Path           # cross-platform file path handling
from transformers import pipeline  # the single Hugging Face function we use throughout
import torch                       # PyTorch; needed only to check for GPU availability

# ── Device selection ──────────────────────────────────────────────────────────
# Hugging Face pipelines accept a `device` argument:
#   0   = first CUDA GPU (fast; use if available)
#  -1   = CPU (always available; 2–4× slower for these model sizes)
#
# torch.cuda.is_available() returns True if a CUDA-capable GPU is present AND
# the PyTorch CUDA build is installed (Colab GPU runtimes satisfy both).
DEVICE = 0 if torch.cuda.is_available() else -1
print("Using GPU" if DEVICE == 0 else "Using CPU")

# ── Dataset directory ─────────────────────────────────────────────────────────
# All five demo datasets live under datasets/ so that file paths are consistent
# whether you upload the folder manually or generate it via the next cell.
DATA_DIR = Path("datasets")
if not DATA_DIR.exists():
    DATA_DIR.mkdir(exist_ok=True)

Using CPU


## Upload the datasets
If you are running this notebook in Colab, upload the accompanying `datasets/` folder or individual files.

If you want, you can also create the demo datasets directly inside the notebook by running the next cell.

> 📌 Why synthetic data?**  
> The five datasets below are intentionally small and fictional.  
> This keeps the notebook self-contained (no external downloads or API keys),  
> makes it safe to share publicly, and gives you full control over the examples  
> so you can craft edge cases that provoke useful classroom discussion.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Create all five demo datasets
# ─────────────────────────────────────────────────────────────────────────────
# Each dataset is written to the datasets/ directory.
# CSVs are used for tabular data (reviews, tickets, emails).
# JSONL (one JSON object per line) is used for the richer text blobs
# (executive updates, policy contexts) where a plain CSV column would be
# hard to read.
#
# ── Dataset 1: Customer reviews ───────────────────────────────────────────────
# Six synthetic reviews across four fictional products and five channels.
# Deliberately include:
#   - clearly positive (R001, R004, R006)
#   - clearly negative (R002)
#   - mixed / ambiguous (R003, R005)  ← good examples for model limits
customer_reviews = pd.DataFrame([
    {"review_id":"R001","product":"SmartHome Hub","channel":"app_store","review_text":"The dashboard is clean and setup was quick. I had my sensors connected in 10 minutes and the automation rules worked perfectly."},
    {"review_id":"R002","product":"SmartHome Hub","channel":"email","review_text":"After the last update the app keeps freezing when I open camera feeds. Very frustrating and not usable during the evening rush."},
    {"review_id":"R003","product":"Fitness+ Subscription","channel":"web","review_text":"Classes are good overall, but billing was confusing and I was charged twice this month."},
    {"review_id":"R004","product":"TravelCard","channel":"social","review_text":"Customer service resolved my lost card issue in one call. Honestly impressed with how fast they handled it."},
    {"review_id":"R005","product":"TravelCard","channel":"survey","review_text":"The exchange rates were okay, but the card got declined twice in Singapore and support chat took forever."},
    {"review_id":"R006","product":"Retail Insights Platform","channel":"g2","review_text":"The analytics module helped our team spot slow-moving stock quickly. Exporting reports is simple and reliable."},
])

# ── Dataset 2: Support tickets ─────────────────────────────────────────────────
# Six tickets from a fictional shared service desk at 'Northwind'.
# Each ticket maps to a plausible routing category, but the text is written
# as a user would naturally write it — no category labels in the text itself.
# This mirrors the zero-shot challenge: the model sees only free text.
support_tickets = pd.DataFrame([
    {"ticket_id":"T001","submitted_by":"sales@northwind.example","priority":"high","ticket_text":"Our sales dashboard stopped loading after yesterday's deployment. Need urgent help before the client meeting."},
    {"ticket_id":"T002","submitted_by":"ops@northwind.example","priority":"medium","ticket_text":"Please add two new starters to the payroll platform and confirm their access permissions."},
    {"ticket_id":"T003","submitted_by":"finance@northwind.example","priority":"high","ticket_text":"Invoice export from the ERP is failing and month-end reconciliation is blocked."},
    {"ticket_id":"T004","submitted_by":"hr@northwind.example","priority":"low","ticket_text":"Can someone explain the new leave approval workflow for managers? We need a quick walkthrough."},
    {"ticket_id":"T005","submitted_by":"it@northwind.example","priority":"medium","ticket_text":"Laptop deliveries for the Brisbane office have the wrong asset tags. Procurement may need to intervene."},
    {"ticket_id":"T006","submitted_by":"marketing@northwind.example","priority":"medium","ticket_text":"We need help connecting campaign leads from HubSpot into the CRM and mapping the fields correctly."},
])

# ── Dataset 3: Executive operational updates ───────────────────────────────────
# Three paragraph-length updates from different departments.
# Each contains a mix of good news, a remaining problem, and a proposed action —
# which mirrors what a real update looks like and gives the summariser something
# meaningful to compress.
executive_updates = [
    {"update_id":"U001","department":"Operations","source_text":"Over the past quarter, the warehouse team reduced average picking time from 14.2 minutes to 11.6 minutes after reorganising the fast-moving inventory zone and introducing handheld scan prompts. The improvement was strongest on Mondays and Tuesdays. However, returns processing remains a bottleneck, with average turnaround still sitting above the target at 39 hours. The team proposes adding a dedicated returns lane and using the same scan prompts for reverse logistics."},
    {"update_id":"U002","department":"Customer Success","source_text":"Churn in the SME segment improved from 4.8 percent to 3.9 percent after the onboarding team rolled out a 30-day check-in sequence. Customers who completed the sequence were more likely to activate at least three product features. The largest remaining risk is delayed technical setup for multi-site customers, especially when procurement approvals take longer than expected. The team recommends a pre-sales implementation checklist and a shared handover template."},
    {"update_id":"U003","department":"Finance","source_text":"Quarterly cash collections improved by 6 percent after automated reminders were introduced for invoices overdue by more than seven days. Bad debt write-offs are stable, but dispute resolution for enterprise accounts still takes too long because supporting documents are scattered across email threads. Finance proposes a central evidence folder for each enterprise account and an SLA for internal responses to billing disputes."},
]

# ── Dataset 4: Procurement emails ─────────────────────────────────────────────
# Four fictional internal emails.  Each one contains several named entities
# (organisations, people, dates, monetary amounts) that a procurement team
# would normally copy into a structured form by hand.
# Note: the off-the-shelf NER model is NOT trained on financial figures, so
# the AUD amounts will be an interesting miss to discuss in class.
procurement_emails = pd.DataFrame([
    {"email_id":"E001","email_text":"Hi team, please rush the renewal for the Adobe Creative Cloud contract with BrightPixel Pty Ltd. The contract value is AUD 18,400 and the current term ends on 30 June 2026. Our contact is Amelia Chen and the invoice should be sent to finance@northwind.example."},
    {"email_id":"E002","email_text":"Can you prepare a purchase order for 25 Lenovo laptops for the Melbourne office? Supplier is TechSource Australia, estimated total is AUD 41,250, and delivery is requested by 15 April 2026."},
    {"email_id":"E003","email_text":"Reminder: the maintenance agreement with Southern Data Hosting was signed by Ravi Perera on 12 February 2026. The annual fee is AUD 9,900 and the service covers the Bendigo backup environment."},
    {"email_id":"E004","email_text":"We received a quote from EcoFleet Mobility for 3 electric pool vehicles. Proposed lease cost is AUD 2,150 per month over 36 months. Please send questions to Mia Rodriguez before Friday."},
])

# ── Dataset 5: Policy contexts ─────────────────────────────────────────────────
# Three short policy excerpts.  Each is short enough for extractive QA to work
# reliably (the model finds spans within the supplied text).  They contain
# specific numbers (dollar thresholds, day limits) so that correct answers are
# easy to verify — useful for evaluating model confidence.
policy_contexts = [
    {"doc_id":"P001","title":"Hybrid Work Policy","context":"Employees may work remotely up to two days per week with manager approval. Team members handling regulated customer data must use the company VPN at all times when outside the office. For reimbursable home office items, employees can claim up to AUD 300 per financial year with receipts submitted through the expense portal within 30 days of purchase."},
    {"doc_id":"P002","title":"Travel and Expenses Policy","context":"Domestic flights under four hours should be booked in economy class unless an approved medical exemption exists. Accommodation should generally not exceed AUD 240 per night in capital cities unless conference rates or market conditions justify a higher amount. Taxi or rideshare costs are reimbursable when public transport is impractical, especially for early morning or late-night travel."},
    {"doc_id":"P003","title":"Procurement Delegations","context":"Managers may approve one-off purchases up to AUD 5,000. Department heads may approve purchases up to AUD 20,000. Any commitment above AUD 20,000 requires director approval and a recorded procurement justification. Multi-year contracts must also be reviewed by legal before signature."},
]

# ── Save to disk ───────────────────────────────────────────────────────────────
# index=False prevents pandas from writing a spurious row-number column.
customer_reviews.to_csv(DATA_DIR / "customer_reviews.csv", index=False)
support_tickets.to_csv(DATA_DIR / "support_tickets.csv", index=False)
procurement_emails.to_csv(DATA_DIR / "procurement_emails.csv", index=False)

# JSONL (JSON Lines) format: one JSON object per line, no surrounding array.
# Useful for streaming large datasets or when rows vary in structure.
with open(DATA_DIR / "executive_updates.jsonl", "w", encoding="utf-8") as f:
    for row in executive_updates:
        f.write(json.dumps(row) + "\n")

with open(DATA_DIR / "policy_contexts.jsonl", "w", encoding="utf-8") as f:
    for row in policy_contexts:
        f.write(json.dumps(row) + "\n")

print("Demo datasets created.")

Demo datasets created.


---
## 1) Customer review triage with sentiment analysis
**Business scenario:** A product or CX team wants to scan incoming customer reviews and quickly separate positive experiences from likely pain points.

**Task type:** Sequence classification / sentiment analysis  
**Suggested model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english`

> 📌 **About the model**  
> **DistilBERT** is a smaller, faster version of BERT — it is 40% smaller and 60% faster while retaining ~97% of BERT's performance on most tasks.  
> It has been *fine-tuned* on the **SST-2** (Stanford Sentiment Treebank) dataset, which consists of movie reviews labelled POSITIVE or NEGATIVE.  
>  
> Key point: the model was fine-tuned on *movie reviews*, not business reviews.  
> This means it generalises well to general-purpose English sentiment, but may struggle with domain-specific language (e.g., billing disputes, SLA terminology).  
> This is a great prompt for discussing **domain shift** — when the distribution of real-world data differs from the training distribution.

> 📌 **What is sentiment analysis?**  
> Sentiment analysis is a **sequence classification** task: the model reads the entire input text and outputs a category label (here: POSITIVE or NEGATIVE) plus a confidence score between 0 and 1.  
> Under the hood, the transformer produces a vector representation of the text, which is passed through a small classification head (a single linear layer) trained to separate the two classes.


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Load the customer reviews dataset
# ─────────────────────────────────────────────────────────────────────────────
# We load the CSV we created above.  Calling .head() at the end of the cell
# renders the first five rows as a formatted table in Jupyter/Colab — a simple
# way for you to confirm the data looks as expected before running the model.
reviews = pd.read_csv(DATA_DIR / "customer_reviews.csv")
reviews.head()

,review_id,product,channel,review_text
0,R001,SmartHome Hub,app_store,The dashboard is clean and setup was quick. I ...
1,R002,SmartHome Hub,email,After the last update the app keeps freezing w...
2,R003,Fitness+ Subscription,web,"Classes are good overall, but billing was conf..."
3,R004,TravelCard,social,Customer service resolved my lost card issue i...
4,R005,TravelCard,survey,"The exchange rates were okay, but the card got..."


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Run sentiment analysis on all reviews
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — specify the model and create the pipeline.
# On first run this downloads the model weights from Hugging Face (~270 MB).
# Subsequent runs in the same session reuse the cached weights.
sentiment_model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
sentiment_pipe = pipeline("sentiment-analysis", model=sentiment_model_id, device=DEVICE)

# Step 2 — run inference on all review texts at once.
# Passing a list lets the pipeline batch the texts together, which is more
# efficient than calling it once per row (especially on GPU).
#
# truncation=True tells the tokeniser to cut off texts longer than the model's
# maximum input length (512 tokens for most BERT-family models).  Without this
# flag a very long review would raise an error.
sentiment_results = sentiment_pipe(reviews["review_text"].tolist(), truncation=True)

# Step 3 — unpack the results into new DataFrame columns.
# Each item in sentiment_results is a dict like:
#   {'label': 'POSITIVE', 'score': 0.9998}
#
# We round the score to 4 decimal places for display clarity.
# A score close to 1.0 = high confidence; close to 0.5 = ambiguous.
reviews["sentiment_label"] = [r["label"] for r in sentiment_results]
reviews["sentiment_score"] = [round(r["score"], 4) for r in sentiment_results]

# Display only the columns relevant to the outcome.
# Note: review_text is included last so you can verify the label makes sense.
reviews[["review_id", "product", "channel", "sentiment_label", "sentiment_score", "review_text"]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu


,review_id,product,channel,sentiment_label,sentiment_score,review_text
0,R001,SmartHome Hub,app_store,POSITIVE,0.9949,The dashboard is clean and setup was quick. I ...
1,R002,SmartHome Hub,email,NEGATIVE,0.9998,After the last update the app keeps freezing w...
2,R003,Fitness+ Subscription,web,NEGATIVE,0.9774,"Classes are good overall, but billing was conf..."
3,R004,TravelCard,social,POSITIVE,0.9995,Customer service resolved my lost card issue i...
4,R005,TravelCard,survey,NEGATIVE,0.9852,"The exchange rates were okay, but the card got..."
5,R006,Retail Insights Platform,g2,POSITIVE,0.9984,The analytics module helped our team spot slow...


### Discussion
- Where would this help? Review triage, brand monitoring, store/app feedback analysis.
- Where could it fail? Mixed sentiment, domain-specific language, sarcasm, billing edge cases, multilingual data.
- Good extension: Add a business rule such as **flag negative reviews above a confidence threshold** for human follow-up.

> 📌 **Note — reading the scores**  
> Notice that R003 and R005 contain *both* a compliment and a complaint.  
> What label did the model assign?  Was it correct?  
> What does a score of, say, 0.72 tell you versus 0.99?  
> This naturally leads to the concept of a **confidence threshold** — only acting on predictions above a certain score, and routing uncertain cases to a human.

> 📌 **Note — two-class limitation**  
> This model outputs only POSITIVE or NEGATIVE.  Real CX workflows often need  
> finer categories (e.g., billing, product quality, delivery, customer service).  
> That's exactly what Use Case 2 (zero-shot classification) is designed to address.


---
## 2) Support-ticket routing with zero-shot classification
**Business scenario:** A shared service desk receives free-text tickets and wants to route them to the right team without training a custom classifier first.

**Task type:** Zero-shot classification  
**Suggested model:** `facebook/bart-large-mnli`

> 📌 **Note — what is zero-shot classification?**  
> In a traditional classifier, you train on labelled examples for each category.  
> **Zero-shot** means the model has never seen these specific categories during training —  
> yet it can still classify text into them.
>  
> The trick: BART-large-MNLI was trained on **Natural Language Inference (NLI)** —  
> given a *premise* and a *hypothesis*, predict whether the hypothesis is entailed, neutral, or contradicted.  
> Zero-shot classification repurposes this: the ticket text becomes the *premise*,  
> and each candidate label (e.g. "Finance operations") is wrapped into a hypothesis like  
> *"This text is about Finance operations."*  
> The model's *entailment* score for each hypothesis becomes the classification score.

> 📌 **Note — business implication**  
> This approach is extremely valuable for prototyping.  
> You can define new routing categories in an afternoon without collecting labelled training data.  
> The trade-off: it is usually less accurate than a fine-tuned classifier once you have enough labels.


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Load the support tickets dataset
# ─────────────────────────────────────────────────────────────────────────────
tickets = pd.read_csv(DATA_DIR / "support_tickets.csv")
tickets.head()

,ticket_id,submitted_by,priority,ticket_text
0,T001,sales@northwind.example,high,Our sales dashboard stopped loading after yest...
1,T002,ops@northwind.example,medium,Please add two new starters to the payroll pla...
2,T003,finance@northwind.example,high,Invoice export from the ERP is failing and mon...
3,T004,hr@northwind.example,low,Can someone explain the new leave approval wor...
4,T005,it@northwind.example,medium,Laptop deliveries for the Brisbane office have...


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Zero-shot classification of support tickets
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — define the candidate routing queues.
# These labels are entirely up to you — the model has not been trained on them.
# Changing these labels changes the routing behaviour without any retraining.
# Try it: swap "IT support" for "Technical issues" and see if predictions change.
candidate_labels = ["IT support", "HR operations", "Finance operations", "Procurement", "CRM / Marketing ops", "Analytics / BI"]

# Step 2 — create the pipeline.
# BART-large-MNLI is larger than DistilBERT (~1.6 GB), so the initial download
# takes longer.  On CPU it also runs slower — about 5–10 seconds per ticket.
zero_shot_model_id = "facebook/bart-large-mnli"
zero_shot_pipe = pipeline("zero-shot-classification", model=zero_shot_model_id, device=DEVICE)

# Step 3 — classify each ticket.
# Unlike the sentiment pipe, we call the zero-shot pipe once per ticket in a loop.
# This is because each call needs its own candidate_labels list.
#
# The output dict contains:
#   result["labels"]  — candidate labels sorted by score, highest first
#   result["scores"]  — corresponding scores (sum to ~1.0 across all labels)
#
# We capture all scores (not just the top-1) so you can see
# the full distribution — useful for discussing confidence and ambiguity.
ticket_predictions = []
for text in tickets["ticket_text"]:
    result = zero_shot_pipe(text, candidate_labels)
    ticket_predictions.append({
        "top_label": result["labels"][0],          # the predicted queue
        "top_score": round(result["scores"][0], 4), # confidence in top label
        "all_labels": result["labels"],             # all labels, ranked
        "all_scores": [round(x, 4) for x in result["scores"]], # all scores
    })

# Step 4 — attach predictions to the DataFrame.
tickets["predicted_queue"] = [r["top_label"] for r in ticket_predictions]
tickets["prediction_score"] = [r["top_score"] for r in ticket_predictions]
tickets[["ticket_id", "priority", "predicted_queue", "prediction_score", "ticket_text"]]

Device set to use cpu


,ticket_id,priority,predicted_queue,prediction_score,ticket_text
0,T001,high,IT support,0.4047,Our sales dashboard stopped loading after yest...
1,T002,medium,Finance operations,0.3041,Please add two new starters to the payroll pla...
2,T003,high,IT support,0.5656,Invoice export from the ERP is failing and mon...
3,T004,low,HR operations,0.5763,Can someone explain the new leave approval wor...
4,T005,medium,Procurement,0.9634,Laptop deliveries for the Brisbane office have...
5,T006,medium,CRM / Marketing ops,0.8148,We need help connecting campaign leads from Hu...


### Discussion
- This is excellent for showing **how far you can get without labeled data**.
- It is useful for prototypes, but production systems usually need label engineering, threshold tuning, and human oversight.
- Great follow-up activity: compare performance when you change the candidate labels.

> 📌 **Note — Label engineering**  
> Look at T005 (wrong asset tags, Brisbane office).  
> It touches both IT and Procurement.  What did the model predict?  
> What happens if you rename "Procurement" to "Asset management"?  
> This demonstrates that label *wording* affects model behaviour — a key insight for practitioners.

> 📌 **Note — confidence and thresholds**  
> Print `ticket_predictions` to show the full ranked list of scores.  
> A ticket with a top score of 0.45 versus 0.95 should be treated very differently.  
> Introduces the idea of a **low-confidence fallback** queue — anything below 0.5 goes to a human for manual triage.


---
## 3) Executive briefing generation with summarization
**Business scenario:** A manager receives long operational updates and wants a short summary that can be dropped into a leadership pack or stand-up briefing.

**Task type:** Abstractive summarization  
**Suggested model:** `sshleifer/distilbart-cnn-12-6`

> 📌 **Note — extractive vs abstractive summarisation**  
> There are two main approaches to summarisation:  
> - **Extractive:** copy the most important sentences directly from the source.  
> - **Abstractive:** generate new sentences that may not appear verbatim in the source.  
>  
> This model uses **abstractive** summarisation, which produces more fluent, readable output  
> but also carries more risk of hallucination or distortion.  
> Discussing this distinction is a natural bridge to LLM hallucination risks.

> 📌 **Note — about the model**  
> `distilbart-cnn-12-6` is a distilled BART model fine-tuned on the CNN/DailyMail news dataset.  
> Its summaries tend to read like news wire leads — concise and declarative.  
> The training data (news articles) is a reasonable but imperfect match for business updates;  
> do you notice any news-style phrasing in the outputs.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Load the executive updates dataset
# ─────────────────────────────────────────────────────────────────────────────
# We use JSONL here because the source_text values are long and would be
# awkward to store or read as CSV columns.
#
# Reading JSONL: open the file, iterate line by line, parse each line as JSON.
# Then convert the list of dicts to a DataFrame.
updates = []
with open(DATA_DIR / "executive_updates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        updates.append(json.loads(line))

updates_df = pd.DataFrame(updates)
updates_df

,update_id,department,source_text
0,U001,Operations,"Over the past quarter, the warehouse team redu..."
1,U002,Customer Success,Churn in the SME segment improved from 4.8 per...
2,U003,Finance,Quarterly cash collections improved by 6 perce...


In [10]:
!pip install "transformers==4.51.3"

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Summarise each executive update
# ─────────────────────────────────────────────────────────────────────────────

# Step 1 — load the summarisation pipeline.
summarizer_model_id = "facebook/bart-large-cnn"
summarizer = pipeline("summarization", model=summarizer_model_id, device=DEVICE)

# Step 2 — summarise each update with length constraints.
#
# max_length=60   — target summary length in tokens (≈ 40–50 words).  Adjust
#                   this if you want shorter or longer summaries.
# min_length=20   — prevents the model from producing a one-word answer.
# do_sample=False — use beam search (deterministic).  Set True for variety,
#                   but outputs will differ on every run.
# truncation=True — cut source text at the model's context limit (1024 tokens
#                   for BART) if it exceeds that.  Our updates are short, so
#                   this will not trigger here — but it is good defensive coding.
summaries = []
for text in updates_df["source_text"]:
    out = summarizer(text, max_length=60, min_length=20, do_sample=False, truncation=True)
    # The pipeline wraps results in a list; [0] gets the first (and only) result.
    # "summary_text" is the key the summarisation pipeline uses for its output.
    summaries.append(out[0]["summary_text"])

# Step 3 — attach summaries and display side by side with source text.
# This side-by-side view is useful for you to judge quality directly.
updates_df["summary"] = summaries
updates_df[["update_id", "department", "summary", "source_text"]]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


,update_id,department,summary,source_text
0,U001,Operations,The warehouse team reduced average picking tim...,"Over the past quarter, the warehouse team redu..."
1,U002,Customer Success,Churn in the SME segment improved from 4.8 per...,Churn in the SME segment improved from 4.8 per...
2,U003,Finance,Cash collections improved by 6 percent after a...,Quarterly cash collections improved by 6 perce...


### Discussion
- This shows **compression of long narrative text into decision-friendly form**.
- Important risk: summaries may omit nuance or distort causality.
- Good governance question: when should a summary be treated as a draft rather than a final business record?

> 📌 **Note — What to look for in outputs**  
> Compare the summary for U001 against the source.  Does the summary mention:  
> - the specific metrics (14.2 → 11.6 min)?  
> - the bottleneck in returns processing?  
> - the proposed solution?  
> Missing information is just as important as what is present.  
> This leads naturally to a discussion of **evaluation metrics** (ROUGE score) and whether  
> a manager could make a good decision from the summary alone.

> 📌 *Note — max_length tuning**  
> Try increasing `max_length` from 60 to 120 and observe how outputs change.  
> Ask: is a longer summary always better?  What is the right length for an exec briefing vs a Slack message?


---
## 4) Contract / procurement extraction with Named Entity Recognition (NER)
**Business scenario:** An operations or procurement team wants to pull key entities out of emails and notes before sending them into a structured workflow.

**Task type:** Token classification / NER  
**Suggested model:** `dslim/bert-base-NER`

> 📌 **Note — what is NER?**  
> Named Entity Recognition (NER) is a **token classification** task.  
> Unlike sentiment analysis (which reads the whole text and outputs one label),  
> NER labels *each token* (roughly each word) in the text with an entity type.  
>  
> The standard CoNLL entity types used by this model are:  
> - **PER** — person name  
> - **ORG** — organisation  
> - **LOC** — location  
> - **MISC** — miscellaneous named entity (events, nationalities, products)
>  
> `aggregation_strategy="simple"` merges consecutive tokens that belong to the same  
> entity into a single span (e.g., "BrightPixel" + "Pty" + "Ltd" → "BrightPixel Pty Ltd").

> 📌 **Note — what this model will miss**  
> The model was not trained to recognise monetary amounts ("AUD 18,400") as a special type —  
> those will likely appear as MISC or be missed entirely.  
> Off-the-shelf NER covers general entities well,  
> but **domain-specific fields require custom labelling or business rules on top**.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Load the procurement emails dataset
# ─────────────────────────────────────────────────────────────────────────────
emails = pd.read_csv(DATA_DIR / "procurement_emails.csv")
emails.head()

,email_id,email_text
0,E001,"Hi team, please rush the renewal for the Adobe..."
1,E002,Can you prepare a purchase order for 25 Lenovo...
2,E003,Reminder: the maintenance agreement with South...
3,E004,We received a quote from EcoFleet Mobility for...


In [13]:
!pip install transformers

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Inspect a single email first
# ─────────────────────────────────────────────────────────────────────────────
# Before running across all emails it is useful to look at the raw entity
# output for a single example.  This helps you understand the structure
# returned by the pipeline before it is wrapped in a helper function.

ner_model_id = "dslim/bert-base-NER"

# aggregation_strategy="simple" merges subword tokens belonging to the same
# entity into a single dict entry, rather than returning one entry per token.
# This is almost always the right setting for downstream use.
ner_pipe = pipeline("ner", model=ner_model_id, aggregation_strategy="simple", device=DEVICE)

example_email = emails.loc[0, "email_text"]
entities = ner_pipe(example_email)

# Each entity dict contains:
#   entity_group — the label (PER, ORG, LOC, MISC)
#   score        — average token-level confidence for this span
#   word         — the surface text of the entity
#   start / end  — character offsets in the original string
entities

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


[{'entity_group': 'ORG',
  'score': np.float32(0.98637885),
  'word': 'Adobe Creative Cloud',
  'start': 41,
  'end': 61},
 {'entity_group': 'ORG',
  'score': np.float32(0.99919486),
  'word': 'BrightPixel Pty Ltd',
  'start': 76,
  'end': 95},
 {'entity_group': 'PER',
  'score': np.float32(0.9990394),
  'word': 'Amelia Chen',
  'start': 188,
  'end': 199}]

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Extract entities from all procurement emails
# ─────────────────────────────────────────────────────────────────────────────

def extract_entities(text):
    """
    Run NER on a single text string and return a compact list of
    (surface_text, entity_type, confidence_score) tuples.

    The tuple format is chosen for display readability in a DataFrame cell.
    In a real system you would return dicts or a structured schema.
    """
    ents = ner_pipe(text)
    return [(e["word"], e["entity_group"], round(e["score"], 4)) for e in ents]

# pandas .apply() calls the function once per row; the result is stored as a
# list of tuples in the new 'entities' column.
emails["entities"] = emails["email_text"].apply(extract_entities)
emails[["email_id", "entities", "email_text"]]

,email_id,entities,email_text
0,E001,"[(Adobe Creative Cloud, ORG, 0.9864), (BrightP...","Hi team, please rush the renewal for the Adobe..."
1,E002,"[(Lenovo, ORG, 0.9979), (Melbourne, LOC, 0.999...",Can you prepare a purchase order for 25 Lenovo...
2,E003,"[(Southern Data Hosting, ORG, 0.9963), (Ravi P...",Reminder: the maintenance agreement with South...
3,E004,"[(EcoFleet Mobility, ORG, 0.9961), (Mia Rodrig...",We received a quote from EcoFleet Mobility for...


### Discussion
- This is a nice bridge into **document intelligence** and **information extraction**.
- Off-the-shelf NER will often miss business-specific fields such as PO numbers, invoice IDs, or internal project codes.
- Good extension: combine NER with regex and business rules.

> 📌 **Note — What entities were missed?**  
> Manually list the entities a procurement officer would want:  
> supplier name, contact name, dollar value, due date, location.  
> Then compare with what the model returned.  
> Dates and dollar values are often partially captured or missed — this is a known gap  
> in CoNLL-trained models and motivates the use of regex for structured fields.

> 📌 **Note — Regex complement example**  
> A simple regex like `r'AUD\s[\d,]+'` would reliably capture dollar amounts that NER misses.  
> The hybrid pattern (NER for fuzzy named entities + regex for structured patterns) is  
> a common production approach worth introducing here.


---
## 5) Internal policy lookup with extractive question answering
**Business scenario:** Staff want fast answers from internal policy text, such as travel, procurement, or hybrid work guidance.

**Task type:** Extractive question answering  
**Suggested model:** `deepset/roberta-base-squad2`

> 📌 **Note — what is extractive QA?**  
> The model is given two inputs: a **question** and a **context** (a passage of text).  
> It predicts the *start* and *end* token positions of the answer span within the context.  
> The answer is therefore always a substring of the supplied context — it cannot make up information.  
>  
> This contrasts with **generative QA** (e.g., GPT-4 answering from memory), where the model  
> generates free text and can hallucinate.  
> Extractive QA is more constrained and therefore easier to audit.

> 📌 **Note — about the model**  
> RoBERTa-base fine-tuned on SQuAD2.  
> SQuAD2 includes *unanswerable* questions (where the answer is not in the context),  
> which means this model is trained to return a low confidence score rather than hallucinate  
> when the context does not contain the answer.  This is a significant practical advantage.

> 📌 **Note — The key limitation**  
> The model only looks at the single context you supply.  
> If the policy is spread across five documents and the answer is in document 3,  
> the model will fail unless you supply document 3.  
> This motivates the next evolution: **Retrieval-Augmented Generation (RAG)**,  
> where a retriever first selects the relevant passage, then the QA model reads it.


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Load the policy contexts dataset
# ─────────────────────────────────────────────────────────────────────────────
policies = []
with open(DATA_DIR / "policy_contexts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        policies.append(json.loads(line))

policies_df = pd.DataFrame(policies)
policies_df

,doc_id,title,context
0,P001,Hybrid Work Policy,Employees may work remotely up to two days per...
1,P002,Travel and Expenses Policy,Domestic flights under four hours should be bo...
2,P003,Procurement Delegations,Managers may approve one-off purchases up to A...


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Single question demo
# ─────────────────────────────────────────────────────────────────────────────
# Run a single question first so you can see the raw output structure
# before the multi-question loop.

qa_model_id = "deepset/roberta-base-squad2"
qa_pipe = pipeline("question-answering", model=qa_model_id, device=DEVICE)

question = "How much can employees claim for home office items each financial year?"

# We explicitly select the P001 context because this question is about the
# Hybrid Work Policy.  Passing the wrong context (e.g., P003) would likely
# result in a low-confidence wrong answer — a good classroom demo.
context = policies_df.loc[0, "context"]

# The QA pipeline expects keyword arguments 'question' and 'context'.
# Output keys: 'answer' (the extracted span), 'score' (confidence), 'start', 'end'.
qa_result = qa_pipe(question=question, context=context)
qa_result

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cpu


{'score': 0.3829091489315033,
 'start': 240,
 'end': 253,
 'answer': 'up to AUD 300'}

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Run a batch of policy questions across multiple documents
# ─────────────────────────────────────────────────────────────────────────────

# Each tuple is (doc_id, question).
# The doc_id tells us which policy context to use — this mirrors a real lookup
# system where a retriever would identify the relevant document first.
sample_questions = [
    ("P001", "How many days per week may employees work remotely?"),
    ("P001", "How much can employees claim for home office items each financial year?"),
    ("P002", "What is the accommodation guideline for capital cities?"),
    ("P003", "Who can approve purchases above AUD 20,000?"),
]

qa_outputs = []
for doc_id, question in sample_questions:
    # Look up the matching context row by doc_id.
    # .iloc[0] extracts the scalar value from the one-row Series.
    context = policies_df.loc[policies_df["doc_id"] == doc_id, "context"].iloc[0]

    result = qa_pipe(question=question, context=context)

    qa_outputs.append({
        "doc_id": doc_id,
        "question": question,
        "answer": result["answer"],    # the extracted text span
        "score": round(result["score"], 4),  # model confidence (0–1)
    })

# Display as a DataFrame for easy reading.
pd.DataFrame(qa_outputs)

,doc_id,question,answer,score
0,P001,How many days per week may employees work remo...,two,0.6256
1,P001,How much can employees claim for home office i...,up to AUD 300,0.3829
2,P002,What is the accommodation guideline for capita...,AUD 240 per night,0.3135
3,P003,"Who can approve purchases above AUD 20,000?",Department heads,0.9205


### Discussion
- This is a clean way to explain the difference between **extractive QA** and chat-style generation.
- It works best when the answer is present in the supplied context.
- Great extension: compare this with retrieval-augmented generation later in the course.

> 📌 **Note — what to explore in class**  
> 1. what happens if you send the P001 question to the P002 context?  
>    (The model will either return a wrong span or a very low confidence score.)  
> 2. Try a question whose answer is *not* in the policy, e.g.  
>    *"What is the policy for working internationally?"* — watch the score drop.  
> 3. The `score` column is the model's confidence.  At what threshold would you trust the answer?  

> 📌 **Note — RAG preview**  
> The manual doc_id lookup above is a simplified stand-in for a real retriever.  
> In a production system you would embed all policy chunks, index them in a vector store,  
> and retrieve the top-k relevant chunks automatically before calling the QA model.  
> This notebook ends at inference; the RAG architecture is a natural next lesson.


---
## 🎓 Interactive Gradio demos

The cells below wrap Use Case 1 (sentiment) and Use Case 5 (policy QA) in
[Gradio](https://www.gradio.app/) interfaces.  You can type their own
text and see predictions live, without touching any code.

> 📌 **Note — why Gradio?**  
> Gradio turns a Python function into a shareable web UI in a few lines.  
> In Colab it renders inline in the notebook output area and also generates a
> temporary public URL (`gradio.live`) that you on other devices can open.  
> This is a low-friction way to make a model feel *real* rather than academic.
>
> The demos below assume `sentiment_pipe` and `qa_pipe` are already loaded
> (i.e. you have run the corresponding cells in Use Cases 1 and 5 above).
> If you want to run the extensions standalone, re-run those pipeline cells first.
>
> Run only one `demo.launch()` at a time — Gradio occupies a port, and launching
> two simultaneously in the same Colab session can cause a port conflict.
> Call `demo.close()` or restart the runtime between launches if needed.


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Install Gradio
# ─────────────────────────────────────────────────────────────────────────────
# Gradio is not installed by default in Colab.  We pin to >=4.0 to ensure
# the modern Blocks API is available (older versions have a different interface).
!pip -q install "gradio>=4.0"

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Gradio demo — Use Case 1 (Sentiment Analysis)
# ─────────────────────────────────────────────────────────────────────────────
# This demo exposes the sentiment pipeline as an interactive text box.
# You type any review text and the model returns a label + confidence.
#
# Good things to ask you to try:
#   - A clearly positive review
#   - A clearly negative review
#   - A mixed review ("great product but terrible delivery")
#   - A sarcastic review ("Oh great, another software update that broke everything")
#   - A review in another language
# Each of these probes a different model limitation.

import gradio as gr

def predict_sentiment(review_text: str) -> str:
    """
    Run the DistilBERT sentiment pipeline on a single user-supplied string
    and return a formatted result string for display.

    Returns a plain string so Gradio renders it in the text output box.
    """
    if not review_text.strip():
        return "⚠️  Please enter some review text."

    # sentiment_pipe was created in Use Case 1 — reuse it here to avoid
    # downloading the model weights a second time.
    result = sentiment_pipe(review_text, truncation=True)[0]

    label = result["label"]           # "POSITIVE" or "NEGATIVE"
    score = round(result["score"], 4) # confidence between 0 and 1

    # Choose an emoji so the output is immediately readable at a glance.
    icon = "✅" if label == "POSITIVE" else "❌"

    # Format a human-friendly summary.
    # The confidence bar (filled vs empty blocks) gives a visual sense of
    # how certain the model is — useful for classroom discussion.
    filled  = int(score * 10)          # e.g. 0.87 → 8 filled blocks
    empty   = 10 - filled
    bar     = "█" * filled + "░" * empty

    return (
        f"{icon}  Label:      {label}\n"
        f"    Confidence: {score}  [{bar}]\n\n"
        f"    Interpretation: {'High confidence' if score > 0.9 else 'Moderate confidence' if score > 0.7 else 'Low confidence — worth reviewing manually'}"
    )

# ── Build the Gradio interface ────────────────────────────────────────────────
# gr.Interface is the simplest Gradio abstraction:
#   fn      — the Python function to call when the user submits
#   inputs  — a Textbox for the review text
#   outputs — a Textbox to display the formatted result
#
# examples= provides clickable sample inputs so you can try the model
# without having to type anything.
sentiment_demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type or paste a customer review here…",
        label="Customer review text",
    ),
    outputs=gr.Textbox(label="Sentiment prediction", lines=5),
    title="Use Case 1 — Customer Review Sentiment",
    description=(
        "Type any product or service review. "
        "The model predicts whether the overall sentiment is POSITIVE or NEGATIVE "
        "and reports how confident it is.  Try mixed or sarcastic reviews to see where it struggles."
    ),
    examples=[
        ["The dashboard is clean and setup was quick. My sensors connected in 10 minutes."],
        ["App keeps freezing after the last update. Very frustrating and basically unusable."],
        ["Classes are good overall, but billing was confusing and I was charged twice."],
        ["Oh great, another software update that broke everything. Really impressive work."],
    ],
    # allow_flagging="never" removes the 'Flag' button.
    allow_flagging="never",
)

# launch() renders the UI inline in the Colab output cell.
# share=True also prints a temporary public gradio.live URL valid for 72 hours,
# so you on phones or other laptops can access the same demo.
sentiment_demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2b8de45ae9774767f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Gradio demo — Use Case 5 (Policy Question Answering)
# ─────────────────────────────────────────────────────────────────────────────
# This demo lets you type a question AND paste or edit the policy context.
# Having both fields editable is intentional:
#   - You can experiment with different questions against the same context.
#   - You can paste a completely different policy to see whether it still works.
#   - You can try a question whose answer is NOT in the context and watch
#     the confidence score drop — note this.

# Pre-populate the context field with the Hybrid Work Policy so the demo is
# immediately usable without any typing.
DEFAULT_CONTEXT = (
    "Employees may work remotely up to two days per week with manager approval. "
    "Team members handling regulated customer data must use the company VPN at all times "
    "when outside the office. For reimbursable home office items, employees can claim up to "
    "AUD 300 per financial year with receipts submitted through the expense portal within "
    "30 days of purchase."
)

def answer_policy_question(question: str, context: str) -> str:
    """
    Run the RoBERTa QA pipeline on the supplied question and context.
    Returns a formatted string showing the extracted answer and confidence.
    """
    if not question.strip():
        return "⚠️  Please enter a question."
    if not context.strip():
        return "⚠️  Please provide a policy context for the model to read."

    # qa_pipe was created in Use Case 5 — reuse without reloading weights.
    result = qa_pipe(question=question, context=context)

    answer = result["answer"]
    score  = round(result["score"], 4)

    # Visual confidence bar — same pattern as the sentiment demo.
    filled = int(score * 10)
    bar    = "█" * filled + "░" * (10 - filled)

    # Classify confidence into three tiers with a plain-English interpretation.
    if score >= 0.80:
        confidence_label = "High — likely reliable"
    elif score >= 0.50:
        confidence_label = "Moderate — verify before acting"
    else:
        confidence_label = "Low — answer may not be in the context"

    return (
        f"📄  Answer:      {answer}\n"
        f"    Confidence: {score}  [{bar}]\n"
        f"    Assessment: {confidence_label}\n\n"
        f"    Note: this model extracts a span directly from the context above.\n"
        f"    It cannot answer questions whose answer is absent from the text."
    )

# ── Build the Gradio interface ────────────────────────────────────────────────
# This demo uses two input fields: one for the question, one for the context.
# Using gr.Blocks (instead of gr.Interface) gives more layout control,
# but gr.Interface is fine here — keep it simple for understanding purposes.
qa_demo = gr.Interface(
    fn=answer_policy_question,
    inputs=[
        gr.Textbox(
            lines=2,
            placeholder="e.g. How much can I claim for home office items?",
            label="Your question",
        ),
        gr.Textbox(
            lines=8,
            value=DEFAULT_CONTEXT,   # pre-filled so the demo works immediately
            label="Policy context (paste any policy text here)",
        ),
    ],
    outputs=gr.Textbox(label="Model answer", lines=7),
    title="Use Case 5 — Policy Question Answering",
    description=(
        "Ask a question about the policy text in the context box. "
        "The model extracts an answer span directly from the text — it cannot invent information. "
        "Try editing the context or asking a question the policy does not cover."
    ),
    examples=[
        ["How many days per week may employees work remotely?",       DEFAULT_CONTEXT],
        ["How much can I claim for home office equipment?",           DEFAULT_CONTEXT],
        ["Do I need to use a VPN when working from home?",           DEFAULT_CONTEXT],
        ["What is the policy for working internationally?",          DEFAULT_CONTEXT],  # unanswerable — score drops
    ],
    allow_flagging="never",
)

# Close the sentiment demo first if it is still running to free the port.
# Comment this line out if you want both demos open simultaneously
# (they will occupy different ports and both remain accessible).
try:
    sentiment_demo.close()
except Exception:
    pass  # sentiment_demo may not exist if this cell is run in isolation

qa_demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e4ceb3deed8b427a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


> 📌 **Suggested activity sequence for the Gradio cells**  
>
> **Sentiment demo:**  
> 1. Run with the pre-loaded positive example — show the high confidence score.  
> 2. Read R003 aloud: *"Classes are good overall, but billing was confusing."* Ask the class to predict the label, then submit it.  
> 3. Try a sarcastic review. Discuss why transformer models struggle with sarcasm.  
> 4. Ask: at what confidence score would you trust this to auto-flag a review without human review?
>
> **Policy QA demo:**  
> 1. Show a high-confidence correct answer (e.g. the AUD 300 home office question).  
> 2. Type a question whose answer is **not** in the policy — watch the score fall below 0.3.  
> 3. Paste in a completely different policy text (e.g. the Travel Policy from P002). Ask the same questions — do they still work?  
> 4. Discuss: what would need to change to make this work across *all* company policies at once? (→ RAG)


---
## Wrap-up table: five authentic business use cases

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Summary comparison table
# ─────────────────────────────────────────────────────────────────────────────
# A single DataFrame that consolidates what you have seen.
# Useful to display on a projector at the end of a session or export to slides.
#
# Each row corresponds to one section above.  The 'Business value' column is
# deliberately written in outcome language (what does a business *get*?) rather
# than technical language.
summary_table = pd.DataFrame([
    {"Use case":"Customer review triage",      "Task":"Sentiment analysis",          "Model":"distilbert/distilbert-base-uncased-finetuned-sst-2-english", "Business value":"Prioritise unhappy customers and track product sentiment"},
    {"Use case":"Support ticket routing",       "Task":"Zero-shot classification",    "Model":"facebook/bart-large-mnli",                                   "Business value":"Route incoming work without labeled training data"},
    {"Use case":"Executive update compression", "Task":"Summarization",               "Model":"sshleifer/distilbart-cnn-12-6",                              "Business value":"Turn long updates into concise leadership summaries"},
    {"Use case":"Procurement email extraction", "Task":"Named Entity Recognition",    "Model":"dslim/bert-base-NER",                                        "Business value":"Extract key entities from business text before workflow entry"},
    {"Use case":"Policy lookup",                "Task":"Question answering",          "Model":"deepset/roberta-base-squad2",                                "Business value":"Answer staff questions from known policy text"},
])
summary_table

,Use case,Task,Model,Business value
0,Customer review triage,Sentiment analysis,distilbert/distilbert-base-uncased-finetuned-s...,Prioritise unhappy customers and track product...
1,Support ticket routing,Zero-shot classification,facebook/bart-large-mnli,Route incoming work without labeled training data
2,Executive update compression,Summarization,sshleifer/distilbart-cnn-12-6,Turn long updates into concise leadership summ...
3,Procurement email extraction,Named Entity Recognition,dslim/bert-base-NER,Extract key entities from business text before...
4,Policy lookup,Question answering,deepset/roberta-base-squad2,Answer staff questions from known policy text


## Suggested classroom prompts
1. Which use case is most production-ready with an off-the-shelf model, and why?
2. Which use case would most likely require domain fine-tuning?
3. Where would you place a human in the loop?
4. What risks appear if you treat high confidence as ground truth?
5. Which scenario should use rules + ML together rather than ML alone?

## Possible extensions
- Add simple evaluation columns and human labels
- Compare multiple Hugging Face models for the same task
- Add threshold-based escalation rules
- Wrap one example in a Gradio demo
- Convert one workflow into a small batch-processing pipeline

---

## 📌 Further Notes - Cross Cutting Themes

**Theme 1 — The pipeline abstraction hides complexity (by design)**  
You will notice that the code for all five use cases looks almost identical.  
That is the point of `pipeline()` — it is a student-friendly abstraction.  
In a follow-up session you can peel back the layers: tokenisation → model forward pass → output decoding.

**Theme 2 — Model cards matter**  
Every model on Hugging Face has a model card listing training data, known biases, and limitations.  
Before using any model in a real system, reading the model card is a professional responsibility.  
A good exercise: open the model card for `dslim/bert-base-NER` and identify one limitation relevant to the procurement scenario.

**Theme 3 — Confidence ≠ correctness**  
A score of 0.99 means the model is very confident — not that it is correct.  
You should be able to construct an example where a high-confidence prediction is clearly wrong.  
This is fundamental to building appropriate human oversight into any AI workflow.

**Theme 4 — The cost of a wrong prediction varies by use case**  
A misclassified support ticket (Use Case 2) causes a minor routing delay.  
A wrong answer from a policy lookup (Use Case 5) about a procurement threshold could cause a compliance breach.  
Rank the five use cases by the cost of an error and discuss what that implies for threshold setting and human review.

**Theme 5 — These models are a starting point, not a finish line**  
All five models here are general-purpose.  
Production systems typically require fine-tuning on domain data, evaluation on held-out examples, and ongoing monitoring for distribution shift.  
This notebook shows the floor; the ceiling is much higher with investment.
